# v15 misclassification analysis — what did the composed detector get wrong?

**v15** is the composed rule-based detector (per-rule events + known-complex-subunit structural rules) measured against Breuer 2019 essentiality across all 455 labeled syn3A loci.

| | Predicted essential | Predicted non-essential |
|---|---|---|
| **Truly essential** (Essential+Quasi) | TP = 287 | FN = 96 |
| **Truly non-essential** | FP = 3 | TN = 69 |

MCC = 0.537 — precision 99 %, recall 75 %. The detector is conservative: it only fires when an annotation rule matches, so **errors are almost entirely missed essentials**, not false alarms.

This notebook loads the saved v15 predictions and breaks the misclassifications down by:
1. Primary Function category
2. Essentiality subtype (Essential vs Quasiessential)
3. Gene-product keywords (uncharacterized, transporter, etc.)
4. Detector failure_mode — to confirm whether the rules silently missed them
5. Detailed listing of every FN and FP with the gene product

Reproduces the run: see the bottom of this notebook for the CLI.

## 1. Setup — clone the branch (Colab)
Skip if already running inside the repo.

In [ ]:
import os, subprocess
REPO   = 'https://github.com/Nikku03/cell.git'
BRANCH = 'claude/vectorize-gex-propensity-NRqBW'
WORK   = '/content/cell'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', REPO, WORK], check=True)
os.chdir(WORK)
subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull',  'origin', BRANCH], check=True)
print('HEAD:', subprocess.check_output(['git','log','-1','--oneline']).decode().strip())

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'openpyxl'], check=True)

## 2. Load v15 predictions + Breuer labels

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
pd.set_option('display.max_colwidth', 70)

V15_TAG = 'parallel_s0.05_t0.5_seed42_thr0.1_w4_composed_all455_v15_round2_priors'
PRED_CSV = Path('outputs') / f'predictions_{V15_TAG}.csv'
BREUER   = Path('memory_bank/data/syn3a_essentiality_breuer2019.csv')

pred = pd.read_csv(PRED_CSV)
br   = pd.read_csv(BREUER)
print(f'predictions: {len(pred)} rows from {PRED_CSV.name}')
print(f'breuer:      {len(br)} rows')

br['y_true'] = br['essentiality'].isin(['Essential', 'Quasiessential']).astype(int)
df = pred.merge(
    br[['locus_tag', 'gene_product', 'essentiality', 'primary_function', 'y_true']],
    on='locus_tag', how='left',
)
df['y_pred'] = df['essential'].astype(int)
df['kind']   = np.where(df.y_true == 1,
                          np.where(df.y_pred == 1, 'TP', 'FN'),
                          np.where(df.y_pred == 1, 'FP', 'TN'))
df.head()

## 3. Confusion + MCC sanity check

In [ ]:
from math import sqrt
counts = df['kind'].value_counts().to_dict()
tp, fp, tn, fn = counts.get('TP', 0), counts.get('FP', 0), counts.get('TN', 0), counts.get('FN', 0)
denom = sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)) or 1
mcc   = (tp*tn - fp*fn) / denom
print(f'TP={tp}  FP={fp}  TN={tn}  FN={fn}    n={tp+fp+tn+fn}')
print(f'precision = {tp/max(tp+fp,1):.3f}')
print(f'recall    = {tp/max(tp+fn,1):.3f}')
print(f'MCC       = {mcc:.3f}    (expected 0.537)')

## 4. Function breakdown of misclassified genes

### 4.1 FN (missed essentials) by `primary_function`

In [ ]:
fn_df = df[df.kind == 'FN'].copy()
print(f'{len(fn_df)} missed essentials.\n')
by_fn = fn_df.groupby('primary_function').agg(
    n=('locus_tag', 'count'),
    essential=('essentiality', lambda s: (s == 'Essential').sum()),
    quasi=('essentiality',     lambda s: (s == 'Quasiessential').sum()),
).sort_values('n', ascending=False)
print(by_fn.to_string())

**Interpretation**: the largest bucket is *Unclear* (functionally uncharacterized) — these genes have no annotation rule available, so the detector cannot fire. *Genetic Information Processing* misses are accessory regulators / rare nucleases. *Metabolism* misses are mostly transporters where redundancy hides essentiality from the rule-based logic.

### 4.2 FN by `failure_mode` (detector signal)
If *every* FN has `failure_mode = none`, the detector produced no signal at all — i.e. it didn't see them as candidates.

In [ ]:
fm = fn_df['failure_mode'].fillna('(none)').value_counts()
print(fm.to_string())

### 4.3 Keyword breakdown of FN gene products

In [ ]:
def categorize(p):
    p = str(p).lower()
    if 'hypothetical' in p or 'uncharacteriz' in p or p == 'nan': return 'uncharacterized / hypothetical'
    if 'transport' in p or 'permease' in p or 'abc' in p: return 'transporter / permease'
    if 'ribosom' in p: return 'ribosomal'
    if 'trna'   in p or 'synthetase' in p: return 'tRNA / aaRS'
    if 'polymerase' in p or 'helicase' in p or 'gyrase' in p: return 'DNA/RNA machinery'
    if 'kinase' in p or 'phosphatase' in p: return 'kinase / phosphatase'
    if 'synthase' in p or 'synthetase' in p: return 'synthase'
    if 'reductase' in p or 'oxidase' in p or 'dehydrogenase' in p: return 'redox enzyme'
    if 'lipoprotein' in p or 'membrane' in p: return 'membrane / lipoprotein'
    if 'ase' in p: return 'enzyme (other)'
    return 'other / annotated'
fn_df['product_category'] = fn_df['gene_product'].apply(categorize)
print(fn_df['product_category'].value_counts().to_string())

### 4.4 Full FN listing (sorted by Primary Function)
Every missed essential, with its product. Use this to spot patterns.

In [ ]:
show = fn_df.sort_values(['primary_function', 'essentiality'])[
    ['locus_tag', 'gene_name', 'essentiality', 'primary_function', 'gene_product']
]
for _, r in show.iterrows():
    nm = '' if pd.isna(r.gene_name) else r.gene_name
    print(f'  {r.locus_tag} {nm:9s} [{r.essentiality:14s}|{r.primary_function[:28]:28s}] {str(r.gene_product)[:55]}')

### 4.5 The 3 false positives (predicted essential, actually non-essential)

In [ ]:
fp_df = df[df.kind == 'FP']
for _, r in fp_df.iterrows():
    nm = '' if pd.isna(r.gene_name) else r.gene_name
    print(f'  {r.locus_tag} {nm:9s} [{r.essentiality:14s}|{r.primary_function}]')
    print(f'      product:  {r.gene_product}')
    print(f'      evidence: {r.evidence}')

All three FPs are intelligible:
- **xseA / xseB (0105 / 0106)** — Exodeoxyribonuclease VII complex. Annotation says "DNA machinery → essential," but syn3A actually tolerates losing it.
- **rpmG / L33 (0930)** — 50S ribosomal protein L33, one of the rare *dispensable* ribosomal proteins. The structural-subunit rule fires on "ribosomal protein," but L33 is the exception.

## 5. Error rate by category
Aggregated tables — useful for picking which features to add next.

In [ ]:
agg_fn = df.groupby('primary_function').agg(
    n=('locus_tag', 'count'),
    n_essential=('y_true', 'sum'),
    n_FP=('kind', lambda s: (s == 'FP').sum()),
    n_FN=('kind', lambda s: (s == 'FN').sum()),
).assign(
    err_rate=lambda d: (d.n_FP + d.n_FN) / d.n,
    recall  =lambda d: 1 - d.n_FN / d.n_essential.replace(0, np.nan),
).sort_values('err_rate', ascending=False)
print(agg_fn.to_string())

In [ ]:
agg_es = df.groupby('essentiality').agg(
    n=('locus_tag', 'count'),
    n_FP=('kind', lambda s: (s == 'FP').sum()),
    n_FN=('kind', lambda s: (s == 'FN').sum()),
).assign(err_rate=lambda d: (d.n_FP + d.n_FN) / d.n)
print(agg_es.to_string())

## 6. What this implies for the next move

- **96 FN, all with `failure_mode = none`** → the rule detector produced no signal at all for these genes. They didn't trip any rule.
- **55 of the 59 Unclear FN are literally "Uncharacterized protein"** → annotation-based features (gene-class keyword grep, FBA-on-SBML) **cannot help these** because there is nothing to annotate.
- The two label-free signals that can reach an uncharacterized gene are:
  1. **Cross-species essentiality** — BLAST against M. genitalium / pneumoniae / mycoides Tn-seq calls. Conservation IS the signal.
  2. **Trajectory ML** (the decoupled XGBoost path in `cell_sim/lgnn/training/xgb_essentiality.py`) — dynamics features don't care about annotation.
- **3 FP** are the textbook dispensable-r-protein and dispensable-DNA-repair-complex cases — not a recall problem, not worth optimizing.

## 7. v15 reproduction CLI

```bash
python scripts/run_sweep_parallel.py \
    --all --workers 4 \
    --scale 0.05 --t-end-s 0.5 --dt-s 0.05 \
    --seed 42 --threshold 0.10 \
    --use-rust \
    --detector composed \
    --ensemble-policy per_rule_with_pool_confirm \
    --min-confidence 0.15 --min-pool-dev 0.02 \
    --min-wt-events 20 \
    --redundancy-drop-threshold 0.30 \
    --redundancy-min-wt-production 20 \
    --enable-imb155-patches
```

Replicates: identical flags with `--seed 1` and `--seed 2`. ~50 min on 4 workers per seed.